In [1]:
#All packages needed to run TwINFER simulation and inference are listed here. 
#If any of them are not installed, please install them using pip or conda env.
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numba
import tqdm
import scipy
import seaborn
import os
import sys
import joblib
from itertools import product
import joblib 
import importlib


# Code to simulate a synthetic GRN and infer the network using TwINFER


## Details about the simulation

### Set this for both running the simulation and before inferring using TwINFER


In [2]:
#Path to TwINFER code repository
import os, sys
path_to_code_repo = os.path.abspath(getattr(sys.modules['__main__'], '__file__', os.getcwd()))


#Common path to data files
path_to_data = f"{path_to_code_repo}/simulation_example_input_data"
path_to_output = f"{path_to_code_repo}/simulation_example_output_data"

base_config = {
    'n_cells': 6000, #Number of cells before division (number of twin pairs)
    'simulation_time_before_division': 100, #The time used to run the initial cells before division. User must set this time to ensure the population reaches steady state [hours]
    'twin_simulation_time_after_division': 48, #The time twin cells are simulated after division and measurements are stored in the output[hours]
    'twin_measurement_resolution': 1, #The time between each measurement of twin cells [hours]. For example, if twin_sampling_duration is 12 and twin_measurement_resolution is 1, the final dataframe will contain hourly measurements for 12 hours (0 is birth).
    "path_to_connectivity_matrix": f"{path_to_data}/connectivity_matrix_A_to_B.txt", #path to the connectivity matrix specifying the GRN to simulate
    "param_csv": f"{path_to_data}/median_parameter.csv", #Path to the parameters for all genes and interaction terms
    "rows_to_use": [[0, 0]], #Rows in the parameter's csv file for each gene. Example - [0,0] will mean use row 0 parameters for both gene 1 and 2. The length should be equal to number of genes in the system. Ensure that each row in the parameter.csv has unique index.
    "output_folder": f"{path_to_output}", #Path to the output folder
    "log_file": f"{path_to_output}/log.jsonl", #Path to the log file
    "type": "A_to_B",  # Name of the network used -- will be in the filename
    "number_of_parallel_parameters": 1, #Number of parameters to be run in parallel
    "number_of_cores_per_parameter": 2, #Number of cores to be used per parameter (number_of_parallel_parameters * number_of_cores_per_parameter = number of cores in your computer)
}

# base_config = {
#         'n_cells': 6000,
#         'simulation_time_before_division': 6000,
#         'twin_simulation_time_after_division': 48,
#         'twin_measurement_resolution': 1,
#         "path_to_connectivity_matrix": "/home/gzu5140/TwINFER-main 3/Matrix/Mutual_regulation.txt",
#         "param_csv": "/home/gzu5140/TwINFER-main 3/simulation_example_input_data/Median_parameter.csv",
#         "rows_to_use": [[0, 1, 2]],
#         "output_folder": "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/three_gene_sim",
#         "log_file": "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/three_gene_sim/logs/Fan_out_median.jsonl",
#         "type": "Five-gene-network",
#         "number_of_parallel_parameters": 1,
#         "number_of_cores_per_parameter": 7
#     }

## Functions and packages used in this notebook - run this everytime notebook is restarted


### For the simulation


In [3]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import set_num_threads, get_num_threads

sys.path.append(str(path_to_code_repo))
set_num_threads(base_config['number_of_cores_per_parameter'])
print("Threads Numba will use:", get_num_threads())

import importlib
from TwINFER_function_scripts import gillespie_script
importlib.reload(gillespie_script)
from TwINFER_function_scripts.gillespie_script import process_param_set

Threads Numba will use: 2


### For inferring with TwINFER


In [4]:
# Calculation functions
import importlib
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers
from TwINFER_function_scripts import infer_with_twinfer

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)
importlib.reload(infer_with_twinfer)

from TwINFER_function_scripts.correlation_analysis_functions import (
    
    calculate_pairwise_gene_gene_correlation_matrix,
    check_system_in_steady_state,
    check_gene_gene_correlation_threshold,
    calculate_twin_random_pair_correlations,
    differentiate_single_state_reg_and_multiple_states,
    identify_reg_if_multiple_states,
    get_cross_correlations,
    identify_actual_directed_edges
)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    extract_param_index,
    read_input_matrix,
    get_param_data, 
    plot_matrix_as_heatmap,
    print_summary,
    plot_network
)

from TwINFER_function_scripts.infer_with_twinfer import (
    infer_with_twinfer
)

## Simulate the gene expression in a population of cells

The code simulates gene expression based on a GRN (described by the interaction matrix) and expression of each gene is defined by parameters (each row in the parameter sheet) using the Gillespie algorithm.


In [ ]:
os.makedirs(base_config['output_folder'], exist_ok=True)
rows_to_use = base_config['rows_to_use']
labels = ["rows_" + "_".join(map(str, row)) for row in rows_to_use]
path_to_simulation_file = process_param_set(rows_to_use[0], labels[0], base_config)
print(f"Saved the simulation file as {path_to_simulation_file}")

# Using TwINFER to infer network from the simulation

### Before starting, set the time-points used as t1 and t2. The default is 1 hour and 20 hours after division.

TwINFER uses three kinds of correlations used for distinguishing multiple-states and regulation.

- First, calculate gene-gene correlations $\rho$ for all pairs of genes at time t1. This will identify gene-pairs that are potentially interacting.
- Second, calculate twin pair correlation $\hat{\rho}_{\Delta}$ and random pair correlation. Comparing these two for all gene pairs will detect the existence of multiple states in the population vs just regulation.
- Finally, if there are multiple states in the population, compare twin pair correlation $\hat{\rho}_{\Delta}$ at time t1 and time t2. This will detect the existence of regulatory interaction.

Next, if there is regulation and single-state in the population, TwINFER can identify direction of regulatory interaction. This is done be comparing cross-correlations between twins at the two timepoints.

### Based on this algorithm, a network will be inferred and plotted.


### Input to TwINFER


### Run TwINFER for inference using the simulation, base_config, t1 and t2.


In [16]:
path_to_simulation_file = f"{path_to_output}/df_rows_0_0_20112025_222419_ncells_6000_A_to_B_4e702629.csv"
twinfer_kwargs = {
    "path_to_simulation_file": path_to_simulation_file,
    "base_config": base_config,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "merge":False,
    "check_for_steady_state": False,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.01, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 10, 
    "p_value_threshold_cross_correlation": 0.01,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": "single-state", #can be either single-state or all-edges,
    "merge": False,
    "n_cores": 4,
    "remove_twin_structure": True
}


In [ ]:
correlation_matrices = infer_with_twinfer(**twinfer_kwargs)